In [1]:
import os
# Set environment variables to avoid deadlocks with dataloaders
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Set this to the GPU you want to use
os.environ["CUDA_VISIBLE_DEVICES"] = "3" 
import torch.nn as nn
import torch
import numpy as np
import json
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR100, Food101, Flowers102, DTD, EuroSAT
from torchvision import transforms
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
from transformers import CLIPModel, CLIPProcessor
from peft import PeftModel
import timm
import random
# --- Prompt Templates for Analysis  ---
from dataset_helpers import GENERAL_TEMPLATES
from dataset_helpers import DTD_TEMPLATES
from dataset_helpers import EUROSAT_TEMPLATES
from dataset_helpers import FLOWERS102_CLASS_NAMES

In [2]:
# --- CONFIGURATION ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = os.path.expanduser("~/.cache")
BATCH_SIZE = 64
DATA_ROOT = os.path.expanduser("~/.cache")
SEED = 42
K_SHOTS = [1, 2, 4, 8, 16]
RESULTS_DIR = "results"
# --- UPDATE THESE PATHS TO MATCH YOUR FILES ---
CLIP_MODEL_ID = "openai/clip-vit-base-patch32"
LORA_ADAPTER_PATH = "clip_lora_checkpoints/epoch-3" # Path to your saved LoRA
FROZEN_CHECKPOINT = "frozen_checkpoints/frozen_checkpoint_epoch1.pt"
# --- REPRODUCIBILITY SETUP ---
def set_seed(seed=42):
    """Sets seeds for Python, NumPy, and PyTorch to ensure reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
print(f"Running comparison on {DEVICE}")

Running comparison on cuda


In [3]:
# ==========================================
# 1. MODEL WRAPPERS
# ==========================================

class CLIPWrapper(nn.Module):
    def __init__(self, base_name, adapter_path=None):
        super().__init__()
        self.base = CLIPModel.from_pretrained(base_name).to(DEVICE)
        self.processor = CLIPProcessor.from_pretrained(base_name)
        
        if adapter_path and os.path.exists(adapter_path):
            print(f"Loading LoRA from {adapter_path}...")
            self.model = PeftModel.from_pretrained(self.base, adapter_path)
            self.name = "LoRA CLIP"
        else:
            self.model = self.base
            self.name = "Standard CLIP"
        self.model.eval()

    def get_features(self, images):
        with torch.no_grad():
            features = self.model.get_image_features(pixel_values=images)
            return features / features.norm(dim=-1, keepdim=True)

    def get_text_features(self, class_names, templates):
        """
        Performs Prompt Ensembling:
        For each class, encode ALL templates (e.g. 80 prompts), then mean-pool them.
        """
        class_embeddings = []
        
        print(f"  Ensembling text prompts for {len(class_names)} classes...")
        with torch.no_grad():
            for class_name in tqdm(class_names, leave=False):
                # 1. Create list of all prompts for this class
                texts = [t.format(class_name) for t in templates]
                
                # 2. Tokenize & Encode
                inputs = self.processor(text=texts, padding=True, return_tensors="pt").to(DEVICE)
                class_feats = self.model.get_text_features(**inputs)
                
                # 3. Normalize & Average
                class_feats /= class_feats.norm(dim=-1, keepdim=True)
                mean_feat = class_feats.mean(dim=0)
                mean_feat /= mean_feat.norm()
                
                class_embeddings.append(mean_feat)
        
        return torch.stack(class_embeddings)

class FrozenWrapper(nn.Module):
    def __init__(self, checkpoint_path):
        super().__init__()
        self.name = "Frozen (ResNet)"
        # Rebuild Architecture
        self.backbone = timm.create_model("resnet50", pretrained=False, num_classes=0, global_pool='')
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.projection = nn.Linear(2048, 2560) 
        
        print(f"Loading Frozen from {checkpoint_path}...")
        if os.path.exists(checkpoint_path):
            ckpt = torch.load(checkpoint_path, map_location=DEVICE)
            state_dict = ckpt.get('vision_encoder_state_dict', ckpt)
            # Clean keys
            bb_spec = {k.replace('backbone.', ''): v for k, v in state_dict.items() if 'backbone' in k}
            proj_spec = {k.replace('projection.', ''): v for k, v in state_dict.items() if 'projection' in k}
            self.backbone.load_state_dict(bb_spec, strict=False)
            self.projection.load_state_dict(proj_spec, strict=False)
            
        self.to(DEVICE)
        self.eval()

    def get_features(self, images):
        with torch.no_grad():
            x = self.backbone(images)
            x = self.pool(x).flatten(1)
            prefix = self.projection(x) 
            # Flatten (Batch, 2, 1280) -> (Batch, 2560)
            return prefix.view(prefix.size(0), -1) / prefix.norm(dim=-1, keepdim=True)

    def get_text_features(self, class_names, templates):
        return None # Not supported

In [4]:
# ==========================================
# 2. EVALUATION UTILS
# ==========================================

def extract_features(dataset, model, preprocess):
    class DSWrapper(torch.utils.data.Dataset):
        def __init__(self, ds): self.ds = ds
        def __len__(self): return len(self.ds)
        def __getitem__(self, i): 
            item = self.ds[i]
            img = item[0] if isinstance(item, tuple) else item['image']
            lbl = item[1] if isinstance(item, tuple) else item['label']
            return preprocess(img.convert("RGB")), int(lbl)

    loader = DataLoader(DSWrapper(dataset), batch_size=BATCH_SIZE, num_workers=4, shuffle=False)
    feats, lbls = [], []
    
    print(f"  Extracting {model.name}...")
    for img, lbl in tqdm(loader, leave=False):
        feats.append(model.get_features(img.to(DEVICE)).cpu())
        lbls.append(lbl)
    return torch.cat(feats).numpy(), torch.cat(lbls).numpy()

def run_evaluation_suite(dataset_name, train_set, test_set, models, preprocess, templates, class_names_override=None):
    # Create Directory
    save_dir = os.path.join(RESULTS_DIR, dataset_name.lower())
    os.makedirs(save_dir, exist_ok=True)
    
    dataset_results = {}
    
    # Determine class names (Prefer override if provided, else use dataset attribute)
    if class_names_override:
        class_names = class_names_override
    elif hasattr(train_set, 'classes'):
        class_names = train_set.classes
    else:
        # Fallback for datasets without clear .classes
        labels = [s[1] for s in train_set] if hasattr(train_set, "__getitem__") else []
        unique_labels = sorted(list(set(labels)))
        class_names = [str(i) for i in unique_labels]

    for model in models:
        model_scores = {"Linear Probe": 0.0, "Zero-Shot": 0.0, "Few-Shot": {}}
        
        # 1. Extract Features
        X_train, y_train = extract_features(train_set, model, preprocess)
        X_test, y_test = extract_features(test_set, model, preprocess)
        
        # 2. Zero-Shot (CLIP only - with Ensembling)
        text_feats = model.get_text_features(class_names, templates)
        if text_feats is not None:
            logits = torch.tensor(X_test).to(DEVICE) @ text_feats.T
            preds = logits.argmax(dim=1).cpu().numpy()
            model_scores["Zero-Shot"] = float(np.mean(preds == y_test) * 100)
            
        # 3. Linear Probe (Full)
        clf = LogisticRegression(C=1.0, max_iter=1000, n_jobs=-1)
        clf.fit(X_train, y_train)
        model_scores["Linear Probe"] = float(clf.score(X_test, y_test) * 100)
        
        # 4. Few-Shot
        for k in K_SHOTS:
            set_seed(SEED)
            indices = []
            for c in np.unique(y_train):
                c_idx = np.where(y_train == c)[0]
                if len(c_idx) >= k: indices.extend(np.random.choice(c_idx, k, replace=False))
            
            if indices:
                clf = LogisticRegression(C=1.0, max_iter=500, n_jobs=-1)
                clf.fit(X_train[indices], y_train[indices])
                model_scores["Few-Shot"][k] = float(clf.score(X_test, y_test) * 100)
        
        dataset_results[model.name] = model_scores
        
    # --- SAVE & PLOT ---
    with open(f"{save_dir}/{dataset_name.lower()}_metrics.json", 'w') as f:
        json.dump(dataset_results, f, indent=4)
        
    # Plot A: Bars
    plt.figure(figsize=(8, 5))
    labels = list(dataset_results.keys())
    lp = [dataset_results[m]["Linear Probe"] for m in labels]
    zs = [dataset_results[m]["Zero-Shot"] for m in labels]
    x = np.arange(len(labels))
    width = 0.35
    plt.bar(x - width/2, lp, width, label='Linear Probe', color='skyblue')
    plt.bar(x + width/2, zs, width, label='Zero-Shot', color='lightgreen')
    plt.xticks(x, labels)   
    plt.ylabel('Accuracy (%)')
    plt.title(f'{dataset_name}: Zero-Shot vs Linear Probe')
    plt.legend()
    plt.savefig(f"{save_dir}/{dataset_name.lower()}_bar_chart.png")
    plt.close()
    
    # Plot B: Few-Shot
    plt.figure(figsize=(8, 5))
    for model_name in labels:
        fs_data = dataset_results[model_name]["Few-Shot"]
        if fs_data:
            shots = sorted([int(k) for k in fs_data.keys()])
            accs = [fs_data[k] for k in shots]
            plt.plot(shots, accs, marker='o', label=model_name)
    plt.xlabel('Shots (k)')
    plt.ylabel('Accuracy (%)')
    plt.title(f'{dataset_name}: Few-Shot Scaling')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(f"{save_dir}/{dataset_name.lower()}_few_shot_curve.png")
    plt.close()
    
    print(f"Finished {dataset_name}. Results in {save_dir}/")

In [5]:
# ==========================================
# 3. MAIN EXECUTION
# ==========================================

# Define Transform
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Initialize Models
models = [
    CLIPWrapper(CLIP_MODEL_ID),
    CLIPWrapper(CLIP_MODEL_ID, LORA_ADAPTER_PATH),
    FrozenWrapper(FROZEN_CHECKPOINT)
]

# Define Datasets Config
# Tuple: (Train_DS, Test_DS, Templates, Class_Name_Override)
dataset_config = {
    "CIFAR100": (
        CIFAR100(root=DATA_ROOT, train=True, download=True),
        CIFAR100(root=DATA_ROOT, train=False, download=True),
        GENERAL_TEMPLATES,
        None
    ),
    "Food101": (
        Food101(root=DATA_ROOT, split='train', download=True),
        Food101(root=DATA_ROOT, split='test', download=True),
        GENERAL_TEMPLATES,
        None
    ),
    "Flowers102": (
        Flowers102(root=DATA_ROOT, split='train', download=True),
        Flowers102(root=DATA_ROOT, split='test', download=True),
        GENERAL_TEMPLATES,
        FLOWERS102_CLASS_NAMES # Important!
    ),
    "DTD": (
        DTD(root=DATA_ROOT, split='train', download=True),
        DTD(root=DATA_ROOT, split='test', download=True),
        DTD_TEMPLATES,
        None
    ),
    "EuroSAT": (
        # EuroSAT needs manual split
        None, None, EUROSAT_TEMPLATES, None
    ) 
}

print("Starting Evaluation Loop...")

Loading LoRA from clip_lora_checkpoints/epoch-3...
Loading Frozen from frozen_checkpoints/frozen_checkpoint_epoch1.pt...


/tmp/ipykernel_45558/645143381.py:62: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(checkpoint_path, map_location=DEVICE)


Files already downloaded and verified
Files already downloaded and verified
Starting Evaluation Loop...


In [6]:
for name, (train_ds, test_ds, templs, c_names) in dataset_config.items():
    print(f"\n=== Processing {name} ===")
    
    # Special Handling for EuroSAT (No standard split)
    if name == "EuroSAT":
        set_seed(SEED)
        full_ds = EuroSAT(root=DATA_ROOT, download=True)
        train_size = int(0.8 * len(full_ds))
        test_size = len(full_ds) - train_size
        train_ds, test_ds = torch.utils.data.random_split(full_ds, [train_size, test_size])
        
    run_evaluation_suite(name, train_ds, test_ds, models, preprocess, templs, c_names)

print("\nAll evaluations complete.")


=== Processing CIFAR100 ===
  Extracting Standard CLIP...


  Extracting Standard CLIP...


  Ensembling text prompts for 100 classes...


  Extracting LoRA CLIP...


  Extracting LoRA CLIP...


...

  Extracting LoRA CLIP...


  Ensembling text prompts for 10 classes...


  Extracting Frozen (ResNet)...


  Extracting Frozen (ResNet)...


Finished EuroSAT. Results in results/eurosat/

All evaluations complete.


In [3]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np

def process_all_datasets(source_dir="final_results", target_dir="new_results"):
    # 1. Create the new target directory
    os.makedirs(target_dir, exist_ok=True)
    print(f"--- Saving all new plots to: {os.path.abspath(target_dir)} ---\n")

    # 2. Check if source directory exists
    if not os.path.exists(source_dir):
        print(f"Error: Source directory '{source_dir}' not found.")
        return

    subfolders = [f for f in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, f))]
    
    if not subfolders:
        print("No dataset folders found in 'results'.")
        return

    for dataset_name in subfolders:
        json_path = os.path.join(source_dir, dataset_name, f"{dataset_name}_metrics.json")
        
        if os.path.exists(json_path):
            print(f"Processing: {dataset_name}")
            generate_beautiful_plot(json_path, dataset_name, target_dir)
        else:
            print(f"Skipping {dataset_name}: JSON not found.")

def generate_beautiful_plot(json_path, dataset_name, target_dir):
    # --- Load Data ---
    with open(json_path, 'r') as f:
        dataset_results = json.load(f)

    labels = list(dataset_results.keys())
    lp_scores = [dataset_results[m].get("Linear Probe", 0) for m in labels]
    
    x = np.arange(len(labels))
    width = 0.6  # Slightly wider bars for a fuller look

    # --- BEAUTIFICATION CONFIGURATION ---
    # A professional color palette (Soft Blue, Coral, Sage Green, Slate)
    colors = ['#5DADE2', '#EC7063', '#58D68D', '#AF7AC5'] 
    # If you have more models than colors, this cycles them safely
    bar_colors = [colors[i % len(colors)] for i in range(len(labels))]

    plt.figure(figsize=(9, 6)) # Slightly larger figure
    
    # 1. Grid lines BEHIND the bars (zorder=0)
    plt.grid(axis='y', linestyle='--', alpha=0.5, zorder=0)

    # 2. Plot Bars (zorder=3 ensures they sit on top of the grid)
    bars = plt.bar(x, lp_scores, width, color=bar_colors, edgecolor='white', linewidth=0.7, zorder=3, label=labels)
    
    # 3. Add Value Labels (Bold and clear)
    plt.bar_label(bars, padding=3, fmt='%.1f', fontsize=11, fontweight='bold', color='#333333')

    # 4. X and Y Axis formatting
    plt.xticks(x, labels, fontsize=11)
    plt.yticks(fontsize=10)
    plt.ylabel('Accuracy (%)', fontsize=12, fontweight='medium')
    
    # 5. Title
    plt.title(f'{dataset_name.upper()}: Linear Probe Performance', fontsize=14, fontweight='bold', pad=15)

    # 6. Remove Top and Right Spines (Borders) for a cleaner look
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#666666')
    ax.spines['bottom'].set_color('#666666')

    # 7. Adjust Y-axis limit to give some headroom for labels
    plt.ylim(0, max(lp_scores) * 1.15) 

    # 8. Add a Legend (Optional, but helps if colors represent specific models consistently)
    # Create custom handles for the legend to match bar colors
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=bar_colors[i], label=labels[i]) for i in range(len(labels))]
    plt.legend(handles=legend_elements, loc='upper right', fontsize=10, frameon=True, framealpha=0.9)

    # --- SAVE ---
    save_name = f"{dataset_name}_lp_chart.png"
    save_path = os.path.join(target_dir, save_name)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300) # dpi=300 makes it high resolution
    plt.close()

# --- EXECUTE ---
process_all_datasets()

--- Saving all new plots to: /sda/usama/Comparison/new_results ---

Processing: cifar100
Processing: flowers102
Processing: food101
Processing: dtd
Processing: eurosat
